# Execução da solução no Google Colab (Etapa 4)

Este notebook roda a solução a partir do repositório, sem código próprio. Ele treina os modelos com registro no MLflow, coloca a versão aprovada em produção, aplica o plano de monitoramento aos dados de 2026 e baixa os resultados.

Para rodar, use Ambiente de execução > Executar tudo. Se o repositório ainda não estiver no GitHub, deixe `REPO_URL` vazio e envie o zip do repositório quando o notebook pedir.

## 1. Código

In [ ]:
import os, sys, zipfile, io, subprocess
from pathlib import Path

REPO_URL = ""          # endereço do repositório no GitHub
APROVADOR = "Equipe do projeto"

if Path("../src/treinar.py").exists():
    os.chdir("..")
if not Path("src/treinar.py").exists():
    if REPO_URL:
        subprocess.run(["git", "clone", REPO_URL, "repo"], check=True)
        os.chdir("repo")
    else:
        from google.colab import files
        print("Envie o zip do repositório")
        enviado = files.upload()
        nome, conteudo = next(iter(enviado.items()))
        zipfile.ZipFile(io.BytesIO(conteudo)).extractall("repo")
        raiz = next(p.parent for p in Path("repo").rglob("config.yaml")).parent
        os.chdir(raiz)
import src          # desliga o envio de dados de uso do MLflow
print("Pasta de trabalho", os.getcwd())

## 2. Ambiente

O Colab já traz quase todas as bibliotecas. Instalamos o MLflow e comparamos as versões com o `requirements.txt`.

In [ ]:
!{sys.executable} -m pip install -q mlflow==3.18.0
import importlib.metadata as im
for linha in open("requirements.txt"):
    if "==" in linha:
        pacote, versao = linha.strip().split("==")
        atual = im.version(pacote)
        print(f"{pacote:15s} esperado {versao:10s} instalado {atual:10s}", "" if atual == versao else "DIFERENTE")

## 3. Versões de código, dados e configuração

In [ ]:
!{sys.executable} -m src.versoes

## 4. Treino com registro no MLflow

Cada run grava parâmetros, métricas, modelo e artefatos. O modelo final entra no registro como candidato.

In [ ]:
!{sys.executable} -m src.treinar
assert Path("resultados/experimentos/registro_experimentos.csv").exists(), "o treino não terminou"

## 5. Aprovação

O candidato só vai para produção com um responsável registrado.

In [ ]:
!{sys.executable} -m src.promover --versao candidato --aprovado-por "{APROVADOR}" --motivo "Escolhido pela regra de validação e confirmado no teste de 2025"

## 6. Monitoramento

Os meses de 2026 chegam como dados novos. O modelo em produção é avaliado contra os limites do plano.

In [ ]:
!{sys.executable} -m src.monitorar

In [ ]:
from IPython.display import Markdown, Image, display
display(Markdown(open("resultados/monitoramento/relatorio_monitoramento.md", encoding="utf-8").read()))
for f in sorted(Path("resultados/monitoramento").glob("fig*.png")):
    display(Image(str(f), width=900))

## 7. O que ficou registrado no MLflow

In [ ]:
import mlflow, pandas as pd
mlflow.set_tracking_uri("sqlite:///mlflow.db")
runs = mlflow.search_runs(experiment_names=["previsao-carga-sul"], order_by=["attributes.start_time ASC"])
cols = ["tags.mlflow.runName", "params.algoritmo", "metrics.MAE_validacao", "metrics.MAE_teste", "metrics.MAPE_teste",
        "tags.versao_dados", "tags.versao_codigo", "tags.versao_config"]
display(runs[[c for c in cols if c in runs]].round(1))
cliente = mlflow.tracking.MlflowClient()
apelidos = cliente.get_registered_model("previsao-carga-sul").aliases
for v in cliente.search_model_versions("name='previsao-carga-sul'"):
    nomes = [a for a, n in apelidos.items() if str(n) == str(v.version)]
    print("versão", v.version, "| apelidos", nomes, "|", v.tags.get("situacao"), "| aprovado por", v.tags.get("aprovado_por"))

## 8. Baixar os resultados

O zip traz os resultados, o histórico do modelo e o banco do MLflow. A interface do MLflow abre com `mlflow ui --backend-store-uri sqlite:///mlflow.db` na pasta em que o treino rodou.

In [ ]:
import shutil
pacote = Path("pacote_etapa4")
if pacote.exists():
    shutil.rmtree(pacote)
for pasta in ("resultados", "governanca", "mlartifacts"):
    shutil.copytree(pasta, pacote / pasta)
shutil.copy("mlflow.db", pacote / "mlflow.db")
arquivo = shutil.make_archive("etapa4_resultados", "zip", pacote)
shutil.rmtree(pacote)
print("Gerado", arquivo)
try:
    from google.colab import files
    files.download(arquivo)
except ImportError:
    pass